# Submission Akhir: Machine Learning Pipeline — Klasifikasi Risiko Penyakit Jantung

**Nama:** Naufal Falah
**Username Dicoding:** `naufal_falah_a`

Notebook ini membangun *machine learning pipeline* end-to-end menggunakan
**TensorFlow Extended (TFX)** dengan **Apache Beam** (`BeamDagRunner`) sebagai
*pipeline orchestrator*. Seluruh artifact komponen disimpan di folder
`naufal_falah_a-pipeline/`, dan model yang lolos evaluasi di-*push* ke
`serving_model/naufal_falah_a-pipeline/`.

Komponen yang dirangkai, berurutan:

| # | Komponen | Peran |
|---|---|---|
| 1 | `CsvExampleGen` | Membaca CSV, memecah train/eval, menyimpan sebagai `tf.Example` |
| 2 | `StatisticsGen` | Menghitung statistik deskriptif tiap fitur |
| 3 | `SchemaGen` | Menyimpulkan skema (tipe, domain, presence) dari statistik |
| 4 | `ExampleValidator` | Mendeteksi anomali data terhadap skema |
| 5 | `Transform` | Feature engineering (`tf.Transform`) yang konsisten antara training & serving |
| 6 | `Trainer` | Melatih model Keras |
| 7 | `Resolver` | Mengambil model *blessed* terakhir sebagai baseline |
| 8 | `Evaluator` | Mengevaluasi model (TFMA) dan memutuskan *blessing* |
| 9 | `Pusher` | Mengekspor model *blessed* ke direktori serving |

Kode pipeline yang sama dalam bentuk skrip tersedia di
[`naufal_falah_a-pipeline.py`](naufal_falah_a-pipeline.py); modul `preprocessing_fn`
dan `run_fn` ada di folder `modules/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Persiapan Environment

Pipeline ini diuji dengan `tfx==1.21.0`, `tensorflow==2.21.0`, dan `apache-beam==2.76.0`
pada **Linux x86_64** (mis. Google Colab).

- **Di Google Colab**: hilangkan komentar pada sel berikut untuk meng-*clone* repo dan
  meng-install dependency. Setelah install selesai, lakukan **Runtime ▸ Restart session**
  lalu jalankan ulang `%cd` sebelum lanjut ke sel berikutnya.
- **Di lokal**: pastikan *virtual environment* sudah aktif dan `requirements.txt` sudah
  ter-install, lalu lewati sel ini.

In [2]:
# --- Hanya untuk Google Colab ---
!git clone https://github.com/naufalfalah/naufal_falah_a-pipeline.git proyek
%cd /content/proyek
!pip install -q -r requirements.txt
!pip uninstall -y -q tensorflow-text
exit()

Cloning into 'proyek'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 50 (delta 4), reused 49 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 585.05 KiB | 2.20 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/proyek
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/4

In [1]:
# --- Hanya untuk Google Colab ---
%cd /content/proyek

/content/proyek


Variabel `TF_USE_LEGACY_KERAS=1` harus di-set **sebelum** TensorFlow di-import pertama kali
agar `tf.keras` mengarah ke Keras 2 (`tf_keras`). Ini dibutuhkan karena `model.save(..., save_format="tf")`
dan `signatures` pada Trainer memakai API SavedModel Keras 2.

In [2]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import glob
import subprocess
import sys

import pandas as pd
import tensorflow as tf
import tensorflow_data_validation as tfdv
import tensorflow_model_analysis as tfma
import tensorflow_transform as tft
import tfx
from tensorflow_data_validation.utils.anomalies_util import load_anomalies_binary
from tfx.components import (
    CsvExampleGen,
    Evaluator,
    ExampleValidator,
    Pusher,
    SchemaGen,
    StatisticsGen,
    Trainer,
    Transform,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.proto import example_gen_pb2, pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

print("TensorFlow :", tf.__version__)
print("TFX        :", tfx.__version__)
print("TFT        :", tft.__version__)
print("TFMA       :", tfma.__version__)
print("TFDV       :", tfdv.__version__)

TensorFlow : 2.21.0
TFX        : 1.21.0
TFT        : 1.21.0
TFMA       : 0.52.0
TFDV       : 1.21.0


## 2. Konfigurasi Pipeline

Semua path bersifat relatif terhadap root proyek, jadi notebook harus dijalankan dari
root proyek (tempat folder `modules/` dan `data_clean/` berada).

- `PIPELINE_ROOT` (`naufal_falah_a-pipeline/`) — dibuat otomatis oleh TFX; berisi artifact
  setiap komponen dan database **ML Metadata** (`metadata/metadata.db`).
- `SERVING_MODEL_DIR` — tujuan `Pusher` untuk model yang lolos evaluasi.
- `modules/transform.py` dan `modules/trainer.py` — kode *user-defined function* untuk
  komponen `Transform` dan `Trainer`.

In [3]:
PIPELINE_NAME = "naufal_falah_a-pipeline"

DATA_ROOT = "data_clean"
TRANSFORM_MODULE_FILE = os.path.join("modules", "transform.py")
TRAINER_MODULE_FILE = os.path.join("modules", "trainer.py")

PIPELINE_ROOT = PIPELINE_NAME
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata", "metadata.db")
SERVING_MODEL_DIR = os.path.join("serving_model", PIPELINE_NAME)

assert os.path.isdir("modules"), "Jalankan notebook dari root proyek (folder modules/ tidak ditemukan)."
assert os.path.isfile(TRANSFORM_MODULE_FILE) and os.path.isfile(TRAINER_MODULE_FILE)
print("Root proyek :", os.getcwd())

Root proyek : /content/proyek


## 3. Dataset & Persiapan Data

Dataset: **[Heart Disease (UCI)](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data)** —
920 baris rekam medis gabungan dari 4 rumah sakit (Cleveland, Hungary, Switzerland, VA Long Beach)
dengan 13 atribut klinis klasik.

Komponen `CsvExampleGen` **tidak** melakukan imputasi, sehingga pembersihan dilakukan **sekali**
di luar graph TFX oleh `modules/data_preparation.py` (pandas):

1. Kolom `id` dan `dataset` (asal rumah sakit) dibuang — bukan fitur klinis dan berpotensi *leakage*.
2. Label asli `num` (0–4, tingkat keparahan) dibinerkan menjadi `target`
   (`0` = sehat, `1` = berisiko penyakit jantung).
3. Fitur numerik yang kosong diisi **median** kolomnya.
4. Fitur kategorikal yang kosong diisi kategori baru `"missing"` (pola *missingness* bisa informatif).

Hasilnya adalah `data_clean/heart_disease_clean.csv` yang sudah tersedia di repo. Sel di bawah
hanya meregenerasinya jika CSV mentah `data/heart_disease_uci.csv` ada.

Fitur final yang dipakai model:

| Jenis | Fitur |
|---|---|
| Numerik (6) | `age`, `trestbps`, `chol`, `thalch`, `oldpeak`, `ca` |
| Kategorikal (7) | `sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal` |
| Label | `target` |

In [4]:
RAW_CSV_PATH = os.path.join("data", "heart_disease_uci.csv")
CLEAN_CSV_PATH = os.path.join(DATA_ROOT, "heart_disease_clean.csv")

if os.path.isfile(RAW_CSV_PATH):
    subprocess.run([sys.executable, os.path.join("modules", "data_preparation.py")], check=True)
else:
    print(f"CSV mentah tidak ditemukan, memakai {CLEAN_CSV_PATH} yang sudah ada.")

df = pd.read_csv(CLEAN_CSV_PATH)
print("Ukuran data :", df.shape)
print("Missing     :", int(df.isna().sum().sum()))
print("Distribusi target:")
print(df["target"].value_counts().rename({0: "0 = sehat", 1: "1 = berisiko"}))
df.head()

Ukuran data : (920, 14)
Missing     : 0
Distribusi target:
target
1 = berisiko    509
0 = sehat       411
Name: count, dtype: int64


,age,trestbps,chol,thalch,oldpeak,ca,sex,cp,fbs,restecg,exang,slope,thal,target
0,63,145.0,233.0,150.0,2.3,0.0,Male,typical angina,True,lv hypertrophy,False,downsloping,fixed defect,0
1,67,160.0,286.0,108.0,1.5,3.0,Male,asymptomatic,False,lv hypertrophy,True,flat,normal,1
2,67,120.0,229.0,129.0,2.6,2.0,Male,asymptomatic,False,lv hypertrophy,True,flat,reversable defect,1
3,37,130.0,250.0,187.0,3.5,0.0,Male,non-anginal,False,normal,False,downsloping,normal,0
4,41,130.0,204.0,172.0,1.4,0.0,Female,atypical angina,False,lv hypertrophy,False,upsloping,normal,0


Definisi fitur disimpan terpusat di `modules/heart_disease_constants.py` supaya nama fitur
konsisten di `transform.py`, `trainer.py`, dan konfigurasi `Evaluator`.

In [5]:
print(open(os.path.join("modules", "heart_disease_constants.py")).read())

"""Definisi fitur untuk pipeline klasifikasi risiko penyakit jantung.

Dipakai bersama oleh transform.py dan trainer.py supaya nama-nama fitur
konsisten di seluruh komponen (Transform, Trainer, Evaluator).
"""

NUMERICAL_FEATURES = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak",
    "ca",
]

CATEGORICAL_FEATURES = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "thal",
]

LABEL_KEY = "target"


def transformed_name(key: str) -> str:
    """Menambahkan suffix `_xf` untuk membedakan fitur hasil Transform."""
    return f"{key}_xf"



## 4. ExampleGen

`CsvExampleGen` membaca seluruh CSV di `data_clean/`, mengubah tiap baris menjadi `tf.train.Example`,
lalu menyimpannya sebagai TFRecord (gzip). Data dibagi menjadi **train : eval = 8 : 2** memakai
`hash_buckets`, sehingga pembagian bersifat deterministik (baris yang sama selalu masuk split yang sama).

In [6]:
output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2),
        ]
    )
)

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output_config)

## 5. StatisticsGen

`StatisticsGen` menghitung statistik deskriptif (jumlah, mean, std, min/max, distribusi nilai,
persentase missing, dsb.) untuk setiap fitur pada tiap split, memakai TensorFlow Data Validation (TFDV).
Statistik ini menjadi dasar bagi `SchemaGen` dan `ExampleValidator`.

In [7]:
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])

## 6. SchemaGen

`SchemaGen` menyimpulkan **skema data** dari statistik split train: tipe tiap fitur
(`INT`, `FLOAT`, `BYTES`), domain nilai kategorikal, dan *presence* (fitur wajib ada atau tidak).
Skema ini dipakai `ExampleValidator` untuk mendeteksi anomali dan `Transform` untuk mem-parsing data.

In [8]:
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])

## 7. ExampleValidator

`ExampleValidator` membandingkan statistik setiap split dengan skema, lalu melaporkan anomali
seperti nilai kategori baru di luar domain, fitur yang hilang, atau tipe data yang tidak sesuai.
Karena data sudah dibersihkan di tahap persiapan, diharapkan tidak ada anomali.

In [9]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)

## 8. Transform

`Transform` menjalankan `preprocessing_fn` dari `modules/transform.py` dengan **tf.Transform**:

- Fitur numerik → `tft.scale_to_z_score` (standarisasi mean 0, std 1).
- Fitur kategorikal → `tft.compute_and_apply_vocabulary` (string → indeks integer;
  `num_oov_buckets=1` menampung kategori yang tidak dikenal saat serving).
- Label `target` di-cast ke `int64`.

Nama fitur hasil transformasi diberi akhiran `_xf`. Keunggulan tf.Transform: statistik
(mean/std, vocabulary) dihitung *full-pass* atas data train dan dibekukan ke dalam
`transform_graph`, sehingga preprocessing saat **training dan serving identik**
(menghindari *training-serving skew*).

In [10]:
print(open(TRANSFORM_MODULE_FILE).read())

"""preprocessing_fn untuk komponen Transform (TFX)."""

import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_transform as tft

from heart_disease_constants import (
    CATEGORICAL_FEATURES,
    LABEL_KEY,
    NUMERICAL_FEATURES,
    transformed_name,
)


def preprocessing_fn(inputs):
    """Feature engineering: scaling untuk fitur numerik, vocabulary untuk kategorikal."""
    outputs = {}

    for key in NUMERICAL_FEATURES:
        outputs[transformed_name(key)] = tft.scale_to_z_score(
            tf.cast(inputs[key], tf.float32)
        )

    for key in CATEGORICAL_FEATURES:
        outputs[transformed_name(key)] = tft.compute_and_apply_vocabulary(
            inputs[key], vocab_filename=key, num_oov_buckets=1
        )

    outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.int64)

    return outputs



In [11]:
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=TRANSFORM_MODULE_FILE,
)

## 9. Trainer

`Trainer` memanggil `run_fn` di `modules/trainer.py` (memakai `GenericExecutor` agar model Keras
dapat dilatih). Arsitektur model:

1. **Input**: 6 fitur numerik (`float32`) langsung dipakai; 7 fitur kategorikal (`int64`)
   masing-masing dilewatkan ke layer `Embedding` (dimensi `min(8, ukuran vocabulary)`).
2. Semua vektor di-*concatenate*, lalu `Dense(64, relu) → Dropout(0.3) → Dense(32, relu) → Dropout(0.3)`.
3. **Output**: `Dense(1, sigmoid)` → probabilitas berisiko penyakit jantung.

Loss `binary_crossentropy`, optimizer Adam (lr 1e-3), metrik `binary_accuracy`, `auc`, `precision`,
`recall`, dengan `EarlyStopping` (monitor `val_auc`, patience 5, `restore_best_weights`).

Signature `serving_default` menerima **raw `tf.Example` terserialisasi** dan menerapkan
`transform_features_layer` di dalamnya, sehingga saat serving klien cukup mengirim data mentah
tanpa preprocessing manual.

In [12]:
print(open(TRAINER_MODULE_FILE).read())

"""run_fn untuk komponen Trainer (TFX) — model klasifikasi risiko penyakit jantung."""

import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_transform as tft
from tensorflow.keras import layers
from tfx.components.trainer.fn_args_utils import FnArgs

from heart_disease_constants import (
    CATEGORICAL_FEATURES,
    LABEL_KEY,
    NUMERICAL_FEATURES,
    transformed_name,
)

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3


def _gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type="GZIP")


def _input_fn(file_pattern, tf_transform_output, batch_size=BATCH_SIZE):
    transformed_feature_spec = tf_transform_output.transformed_feature_spec().copy()
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transformed_feature_spec,
        reader=_gzip_reader_fn,
        label_key=transformed_name(LABEL_KEY),
    )
    re

In [13]:
trainer = Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=1000),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=200),
)

## 10. Resolver

`Resolver` dengan strategi `LatestBlessedModelStrategy` mencari model terakhir yang sudah
*blessed* di ML Metadata untuk dijadikan **baseline** bagi `Evaluator`. Pada run pertama belum
ada model blessed, sehingga Resolver mengembalikan kosong dan `Evaluator` hanya memakai
*value threshold* (tanpa perbandingan terhadap baseline).

In [14]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("Latest_blessed_model_resolver")

## 11. Evaluator

`Evaluator` mengevaluasi model dengan **TensorFlow Model Analysis (TFMA)** pada data eval mentah
(`example_gen.outputs["examples"]`) — preprocessing dilakukan oleh signature model sendiri.

`EvalConfig`:

- `ModelSpec`: label `target`, signature `serving_default`.
- `SlicingSpec`: metrik dihitung untuk keseluruhan data **dan** per nilai fitur `sex`
  (memeriksa apakah performa timpang antar jenis kelamin).
- Metrik: `ExampleCount`, `BinaryAccuracy`, `Precision`, `Recall`, `AUC`.
- **Threshold** pada `AUC`: model dinyatakan *blessed* hanya jika AUC ≥ **0.65** (`value_threshold`)
  dan tidak lebih buruk dari baseline lebih dari 0.001 (`change_threshold`, berlaku jika baseline ada).

Hanya model yang *blessed* yang akan diteruskan ke `Pusher`.

In [15]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(label_key="target", signature_name="serving_default")
    ],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=["sex"]),
    ],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name="ExampleCount"),
                tfma.MetricConfig(class_name="BinaryAccuracy"),
                tfma.MetricConfig(class_name="Precision"),
                tfma.MetricConfig(class_name="Recall"),
                tfma.MetricConfig(
                    class_name="AUC",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.65}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={"value": -1e-3},
                        ),
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)

## 12. Pusher

`Pusher` menyalin SavedModel ke `serving_model/naufal_falah_a-pipeline/<timestamp>/` **hanya jika**
`Evaluator` memberikan *blessing*. Direktori inilah yang nantinya dimuat oleh TensorFlow Serving
(lihat `Dockerfile` untuk deployment).

In [16]:
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)

## 13. Merangkai & Menjalankan Pipeline dengan Apache Beam

Seluruh komponen dirangkai dalam `tfx.orchestration.pipeline.Pipeline`. Urutan eksekusi
ditentukan otomatis dari dependensi *input/output* antar komponen.

- **Orchestrator**: `BeamDagRunner`, yang menjalankan setiap komponen sebagai job **Apache Beam**
  (`DirectRunner`, mode *in-memory*, 1 worker).
- **ML Metadata**: SQLite di `naufal_falah_a-pipeline/metadata/metadata.db` — mencatat lineage
  setiap artifact dan eksekusi.
- `enable_cache=True`: jika notebook dijalankan ulang tanpa perubahan input, komponen yang sudah
  pernah sukses tidak dieksekusi ulang. Untuk run bersih hapus dulu folder
  `naufal_falah_a-pipeline/` dan `serving_model/`.

Proses ini memakan waktu sekitar 10–20 menit di CPU (tahap terlama: `Transform` dan `Trainer`).

In [17]:
components = [
    example_gen,
    statistics_gen,
    schema_gen,
    example_validator,
    transform,
    trainer,
    model_resolver,
    evaluator,
    pusher,
]

tfx_pipeline = pipeline.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    components=components,
    enable_cache=True,
    metadata_connection_config=metadata.sqlite_metadata_connection_config(
        METADATA_PATH
    ),
    beam_pipeline_args=[
        "--direct_running_mode=in_memory",
        "--direct_num_workers=1",
    ],
)

In [18]:
BeamDagRunner().run(tfx_pipeline)

Instructions for updating:
Use `tf.data.Dataset.map(tf.io.parse_example(...))` instead.


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 sex_xf (InputLayer)         [(None, 1)]                  0         []                            
                                                                                                  
 cp_xf (InputLayer)          [(None, 1)]                  0         []                            
                                                                                                  
 fbs_xf (InputLayer)         [(None, 1)]                  0         []                            
                                                                                                  
 restecg_xf (InputLayer)     [(None, 1)]                  0         []                            
                                                                                              

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


## 14. Verifikasi Artifact Pipeline

Setelah pipeline selesai, `naufal_falah_a-pipeline/` harus berisi satu subfolder per komponen,
folder `metadata/` (MLMD), dan `_wheels/` (paket modul `Transform`/`Trainer`). Berkas `BLESSED`
di `Evaluator/blessing/` menandakan model lolos threshold dan `Pusher` sudah mengekspor model.

In [19]:
print("Isi", PIPELINE_ROOT, ":", sorted(os.listdir(PIPELINE_ROOT)))
print()

blessing_files = glob.glob(os.path.join(PIPELINE_ROOT, "Evaluator", "blessing", "*", "*"))
print("Berkas blessing :", [os.path.basename(f) for f in blessing_files] or "belum ada")

pushed_models = sorted(glob.glob(os.path.join(SERVING_MODEL_DIR, "*")))
print("Model di-push   :", pushed_models or "tidak ada (model tidak blessed)")

Isi naufal_falah_a-pipeline : ['CsvExampleGen', 'Evaluator', 'ExampleValidator', 'Pusher', 'SchemaGen', 'StatisticsGen', 'Trainer', 'Transform', '_wheels', 'metadata']

Berkas blessing : ['BLESSED']
Model di-push   : ['serving_model/naufal_falah_a-pipeline/1789409415', 'serving_model/naufal_falah_a-pipeline/1789416590']


### 14.1 Statistik, Skema, dan Anomali Data

Menampilkan keluaran `StatisticsGen`, `SchemaGen`, dan `ExampleValidator` memakai utilitas TFDV.
Skema mencantumkan tipe dan domain tiap fitur; tabel anomali diharapkan kosong
(`No anomalies found`).

In [20]:
def latest_artifact_dir(component, output_name):
    dirs = sorted(
        glob.glob(os.path.join(PIPELINE_ROOT, component, output_name, "*")),
        key=lambda p: int(os.path.basename(p)) if os.path.basename(p).isdigit() else -1,
    )
    return dirs[-1]


stats_dir = latest_artifact_dir("StatisticsGen", "statistics")
schema_dir = latest_artifact_dir("SchemaGen", "schema")
anomalies_dir = latest_artifact_dir("ExampleValidator", "anomalies")

train_stats = tfdv.load_stats_binary(
    os.path.join(stats_dir, "Split-train", "FeatureStats.pb")
)
eval_stats = tfdv.load_stats_binary(
    os.path.join(stats_dir, "Split-eval", "FeatureStats.pb")
)
tfdv.visualize_statistics(
    lhs_statistics=train_stats, rhs_statistics=eval_stats,
    lhs_name="TRAIN", rhs_name="EVAL",
)

In [21]:
schema = tfdv.load_schema_text(os.path.join(schema_dir, "schema.pbtxt"))
tfdv.display_schema(schema)

,Type,Presence,Valency,Domain
Feature name,,,,
'age',INT,required,,-
'ca',FLOAT,required,,-
'chol',FLOAT,required,,-
'cp',STRING,required,,'cp'
'exang',STRING,required,,'exang'
'fbs',STRING,required,,'fbs'
'oldpeak',FLOAT,required,,-
'restecg',STRING,required,,'restecg'
'sex',STRING,required,,'sex'


,Values
Domain,
'cp',"'asymptomatic', 'atypical angina', 'non-anginal', 'typical angina'"
'exang',"'False', 'True', 'missing'"
'fbs',"'False', 'True', 'missing'"
'restecg',"'lv hypertrophy', 'missing', 'normal', 'st-t abnormality'"
'sex',"'Female', 'Male'"
'slope',"'downsloping', 'flat', 'missing', 'upsloping'"
'thal',"'fixed defect', 'missing', 'normal', 'reversable defect'"


In [22]:
for split in ("train", "eval"):
    anomalies = load_anomalies_binary(
        os.path.join(anomalies_dir, f"Split-{split}", "SchemaDiff.pb")
    )
    print(f"Anomali split {split}:")
    tfdv.display_anomalies(anomalies)

Anomali split train:


Anomali split eval:


### 14.2 Hasil Evaluasi Model (TFMA)

Memuat hasil `Evaluator` dan menampilkan metrik untuk slice keseluruhan (`Overall`) serta
per nilai `sex`. Nilai `AUC` pada slice keseluruhan inilah yang dibandingkan dengan threshold 0.65.

In [23]:
eval_dir = latest_artifact_dir("Evaluator", "evaluation")
eval_result = tfma.load_eval_result(eval_dir)

rows = []
for slice_key, metrics in eval_result.get_metrics_for_all_slices().items():
    row = {"slice": "Overall" if not slice_key else str(slice_key)}
    for name, value in metrics.items():
        if "doubleValue" in value:
            row[name] = round(value["doubleValue"], 4)
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index("slice")
metrics_df

,binary_accuracy,auc,precision,recall,loss,example_count
slice,,,,,,
"(('sex', 'Female'),)",0.8780,0.9286,0.8462,0.7857,0.3271,41.0
Overall,0.8804,0.9342,0.8899,0.9065,0.3184,184.0
"(('sex', 'Male'),)",0.8811,0.9247,0.8958,0.9247,0.3159,143.0


In [24]:
validation = tfma.load_validation_result(eval_dir)
print("Model blessed (lolos threshold):", validation.validation_ok)

Model blessed (lolos threshold): True


## 15. Uji Coba Model Hasil Pusher

Memuat SavedModel dari `serving_model/` dan mengirim beberapa baris data mentah sebagai
`tf.Example` terserialisasi ke signature `serving_default` — persis seperti yang akan dilakukan
TensorFlow Serving di cloud. Tipe tiap fitur diambil dari `raw_feature_spec` `transform_graph`
agar konsisten dengan skema pipeline.

In [25]:
transform_graph_dir = latest_artifact_dir("Transform", "transform_graph")
raw_feature_spec = tft.TFTransformOutput(transform_graph_dir).raw_feature_spec()
raw_feature_spec.pop("target", None)


def row_to_serialized_example(row):
    feature = {}
    for key, spec in raw_feature_spec.items():
        value = row[key]
        if spec.dtype == tf.int64:
            feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))
        elif spec.dtype == tf.float32:
            feature[key] = tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))
        else:
            feature[key] = tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[str(value).encode("utf-8")])
            )
    return tf.train.Example(features=tf.train.Features(feature=feature)).SerializeToString()


model_dir = sorted(glob.glob(os.path.join(SERVING_MODEL_DIR, "*")))[-1]
loaded_model = tf.saved_model.load(model_dir)
predict_fn = loaded_model.signatures["serving_default"]

sample = df.sample(n=5, random_state=42)
serialized = tf.constant([row_to_serialized_example(r) for _, r in sample.iterrows()])
probabilities = next(iter(predict_fn(examples=serialized).values())).numpy().ravel()

hasil = sample[["age", "sex", "cp", "thalch", "oldpeak", "target"]].copy()
hasil["prob_berisiko"] = probabilities.round(4)
hasil["prediksi"] = (probabilities >= 0.5).astype(int)
hasil

,age,sex,cp,thalch,oldpeak,target,prob_berisiko,prediksi
319,36,Male,atypical angina,180.0,0.0,0,0.0465,0
377,45,Male,atypical angina,122.0,0.0,0,0.0565,0
538,48,Male,asymptomatic,92.0,1.5,1,0.9586,1
296,59,Male,asymptomatic,90.0,1.0,1,0.9807,1
531,40,Female,asymptomatic,130.0,2.0,1,0.8682,1


## 16. Ringkasan

- Sembilan komponen TFX (`CsvExampleGen` → `StatisticsGen` → `SchemaGen` → `ExampleValidator` →
  `Transform` → `Trainer` → `Resolver` → `Evaluator` → `Pusher`) berhasil dirangkai dan dijalankan
  dengan **Apache Beam** melalui `BeamDagRunner`.
- Artifact dan metadata tersimpan di `naufal_falah_a-pipeline/`; model yang lolos threshold
  AUC ≥ 0.65 diekspor ke `serving_model/naufal_falah_a-pipeline/`.
- Model siap disajikan dengan TensorFlow Serving (Dockerfile) dan dipantau dengan Prometheus —
  dijelaskan lebih lanjut di berkas Markdown dokumentasi proyek.

In [26]:
# Export serving_model
!zip -qr hasil_pipeline.zip naufal_falah_a-pipeline serving_model run.log
from google.colab import files; files.download("hasil_pipeline.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>